Milestone 1, Block 1

Identify the galvanised structural members in an Autodesk Inventor HVAC assembly, recognise each member's cross-section, find where members meet, and classify each joint. Output is JSON that the later milestones (cost, manufacturability, capacity) consume without re-opening Inventor.



**Pipeline**

`Inventor session → GLV BOM filter → member geometry → joint detector → joint classifier → JSON`


## 1. Schema

The data model shared across every stage and serialised to JSON for Blocks 2–4. The two joint taxonomies (tubular `T/Y/X/K/N` and open-section `corner/tee/splice/gusset/crossing`) are kept deliberately separate.

In [ ]:
"""Shared data model for Milestone 1, Block 1.

These dataclasses are the *contract* between this block and the later
milestones (cost, manufacturability, capacity). Everything this block
discovers about the model is serialized to JSON through these types, so
downstream code never has to re-open Inventor.

Coordinates are in Inventor's internal units (centimetres) and the
assembly coordinate frame unless noted otherwise.
"""

from __future__ import annotations

from dataclasses import dataclass, field, asdict
from enum import Enum
from typing import Optional
import json


class SectionType(str, Enum):
    """Cross-section families the classifier can recognise.

    Start: CHANNEL (the lipped/plain cee in the reference screenshot).
    The rest are scaffolded so adding a signature later does not change
    any downstream code.
    """
    CHANNEL = "channel"            # open C / cee section (web + 2 flanges)
    ANGLE = "angle"               # L section
    I_BEAM = "i_beam"             # I / W / wide-flange
    TEE = "tee"                   # T section
    PLATE = "plate"               # solid rectangular bar / flat plate
    RECT_HSS = "rect_hss"         # closed rectangular / square hollow section
    ROUND_HSS = "round_hss"       # closed circular hollow section (pipe)
    ROUND_BAR = "round_bar"       # solid round bar
    UNKNOWN = "unknown"


class SectionFamily(str, Enum):
    """Whether the section encloses a void. Drives which joint taxonomy
    applies later (tubular vs open-section connection)."""
    OPEN = "open"                 # channel, angle, I, tee, plate, round bar
    HOLLOW = "hollow"             # rect_hss, round_hss


# Which section types are hollow (tubular). Used by the joint classifier.
HOLLOW_TYPES = {SectionType.RECT_HSS, SectionType.ROUND_HSS}


class JointType(str, Enum):
    """Two taxonomies live here, kept deliberately separate.

    Tubular taxonomy (CIDECT / Eurocode 3) -- ONLY applied when the
    members meeting at the joint are hollow sections:
        T, Y, X, K, N
    Open-section connection-configuration taxonomy -- applied when the
    members are open sections (channels, angles, ...):
        corner, tee_connection, splice, gusset, crossing
    """
    # tubular
    T = "T"
    Y = "Y"
    X = "X"
    K = "K"
    N = "N"
    # open-section connection configuration
    CORNER = "corner"                 # two members, ~mitre / L at their ends
    TEE_CONNECTION = "tee_connection"  # one member ends on the side of another
    SPLICE = "splice"                 # two collinear members joined end-to-end
    GUSSET = "gusset"                 # 3+ members converging (plate-type node)
    CROSSING = "crossing"             # members cross / pass through
    UNKNOWN = "unknown"


@dataclass
class CrossSection:
    section_type: SectionType = SectionType.UNKNOWN
    family: SectionFamily = SectionFamily.OPEN
    # nominal overall envelope of the profile, profile-plane units
    depth: Optional[float] = None        # extent along the section's major axis
    width: Optional[float] = None        # extent along the section's minor axis
    wall_thickness: Optional[float] = None   # approx., for thin-walled sections
    is_lipped: Optional[bool] = None     # channels: returns at flange tips?
    n_loops: Optional[int] = None        # 1 = solid/open, 2 = hollow (void)
    occupancy_signature: Optional[str] = None  # e.g. "111/100/111" (diagnostic)
    detection_method: str = "geometry"   # "geometry" | "extrude_profile"
    confidence: float = 0.0              # 0..1


@dataclass
class Member:
    occurrence_name: str                 # Inventor occurrence path/name
    bom_description: str = ""
    part_number: str = ""
    material: str = ""
    is_glv: bool = False
    # centreline in assembly coordinates
    start_point: tuple[float, float, float] = (0.0, 0.0, 0.0)
    end_point: tuple[float, float, float] = (0.0, 0.0, 0.0)
    length: float = 0.0
    cross_section: CrossSection = field(default_factory=CrossSection)


@dataclass
class Joint:
    joint_id: str
    location: tuple[float, float, float]
    member_names: list[str] = field(default_factory=list)
    # role per member at this node: "chord" | "brace" | "leg" | "" (unknown)
    member_roles: dict[str, str] = field(default_factory=dict)
    angles_deg: list[float] = field(default_factory=list)  # between members
    gap: Optional[float] = None          # K/N gap (tubular) or end gap (open)
    joint_type: JointType = JointType.UNKNOWN
    taxonomy: str = "open"               # "tubular" | "open"
    is_inferred: bool = False            # True = "should go", False = "is there"
    confidence: float = 0.0


@dataclass
class AnalysisResult:
    source_document: str = ""
    members: list[Member] = field(default_factory=list)
    joints: list[Joint] = field(default_factory=list)

    def to_json(self, indent: int = 2) -> str:
        def encode(o):
            if isinstance(o, Enum):
                return o.value
            raise TypeError(f"not serializable: {type(o)}")
        return json.dumps(asdict(self), indent=indent, default=encode)


## 2. Section classifier

Pure Python, no ML. Input is the profile loops of an extruded member (the extrude's sketch *is* the cross-section, since the model uses plain extrudes). A circle test catches round/pipe; everything else is identified by a 3×3 material-occupancy signature matched over all rotations/reflections:

`plate 111/111/111 · rect_hss 111/101/111 · i_beam 111/010/111 · channel 111/100/111 · tee 010/010/111 · angle 100/100/111`

Channels are then checked for return lips. Adding a section type later = adding one signature.

In [12]:
"""Cross-section classifier.

Pure Python. Input is a set of 2D loops (the profile of an extruded
member, as read from its sketch). Output is a CrossSection.

Strategy (deterministic, explainable -- no ML, no training data):

  1. Circle test. Fit a circle to the outer loop. A clean fit means a
     round family: round_bar (1 loop) or round_hss / pipe (2 loops).

  2. Occupancy signature. For non-round profiles, lay a 3x3 grid over
     the profile's principal-axis-aligned bounding box and mark each
     cell that contains ANY material (dense sub-sampling, so thin
     flanges register). That 3x3 pattern is matched -- over all 8
     rotations/reflections -- against one canonical signature per
     section type:

         plate      111/111/111
         rect_hss   111/101/111   (and n_loops == 2)
         i_beam     111/010/111
         channel    111/100/111
         tee        111/010/010
         angle      100/100/111

     Matching over the dihedral group makes orientation irrelevant: a
     channel opening in any of four directions still matches.

  3. Lip refinement. A profile classified as a channel is checked for
     short return segments at the flange tips -> is_lipped.

Why occupancy and not corner-counting: fillets, lips and modelling
noise change vertex counts wildly, but the coarse material layout is
stable. The signature is what an engineer would read off a sketch at a
glance.
"""

from __future__ import annotations

import math
from typing import Sequence


# A loop is a closed list of (u, v) points (arcs already tessellated).
Loop = Sequence[tuple[float, float]]


# One canonical 3x3 signature per recognised type. Rows are top -> bottom.
_CANONICAL = {
    SectionType.PLATE:    ("111", "111", "111"),
    SectionType.RECT_HSS: ("111", "101", "111"),
    SectionType.I_BEAM:   ("111", "010", "111"),
    SectionType.CHANNEL:  ("111", "100", "111"),
    SectionType.TEE:      ("111", "010", "010"),
    SectionType.ANGLE:    ("100", "100", "111"),
}


# ---------------------------------------------------------------------------
# geometry helpers
# ---------------------------------------------------------------------------

def _bbox(points: Loop) -> tuple[float, float, float, float]:
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    return min(xs), min(ys), max(xs), max(ys)


def _point_in_loop(x: float, y: float, loop: Loop) -> bool:
    """Ray-casting point-in-polygon for a single closed loop."""
    inside = False
    n = len(loop)
    j = n - 1
    for i in range(n):
        xi, yi = loop[i]
        xj, yj = loop[j]
        if ((yi > y) != (yj > y)) and (
            x < (xj - xi) * (y - yi) / (yj - yi + 1e-12) + xi
        ):
            inside = not inside
        j = i
    return inside


def _point_in_region(x: float, y: float, outer: Loop, holes: list[Loop]) -> bool:
    """Inside the material: inside the outer loop and not in any hole."""
    if not _point_in_loop(x, y, outer):
        return False
    return not any(_point_in_loop(x, y, h) for h in holes)


def _min_area_angle(outer: Loop) -> float:
    """Orientation that minimises the bounding-box area.

    For rolled/formed sections this snaps the web and flanges onto the
    grid axes (the section sits squarely in its tightest box), which is
    exactly what the occupancy signature needs. PCA does NOT do this --
    an equal-leg angle's principal axis lies on its diagonal -- which is
    why we use the rotating-calipers idea instead. The minimum-area
    rectangle of a polygon always has a side collinear with one of its
    edges, so trying every edge direction is sufficient.
    """
    best_ang, best_area = 0.0, float("inf")
    n = len(outer)
    for i in range(n):
        x1, y1 = outer[i]
        x2, y2 = outer[(i + 1) % n]
        dx, dy = x2 - x1, y2 - y1
        if math.hypot(dx, dy) < 1e-9:
            continue
        ang = math.atan2(dy, dx)
        rot = _rotate(outer, ang)
        minx, miny, maxx, maxy = _bbox(rot)
        area = (maxx - minx) * (maxy - miny)
        if area < best_area:
            best_area, best_ang = area, ang
    return best_ang


def _rotate(points: Loop, ang: float) -> list[tuple[float, float]]:
    c, s = math.cos(ang), math.sin(ang)
    return [(p[0] * c + p[1] * s, -p[0] * s + p[1] * c) for p in points]


def _fit_circle(loop: Loop) -> tuple[float, float, float, float]:
    """Least-squares circle fit. Returns (cx, cy, r, rms_error/r)."""
    n = len(loop)
    sx = sum(p[0] for p in loop)
    sy = sum(p[1] for p in loop)
    cx, cy = sx / n, sy / n
    r = sum(math.hypot(p[0] - cx, p[1] - cy) for p in loop) / n
    if r < 1e-9:
        return cx, cy, r, 1.0
    err = math.sqrt(sum((math.hypot(p[0] - cx, p[1] - cy) - r) ** 2 for p in loop) / n)
    return cx, cy, r, err / r


# ---------------------------------------------------------------------------
# occupancy signature
# ---------------------------------------------------------------------------

def _occupancy(outer: Loop, holes: list[Loop]) -> tuple[tuple[str, str, str], float]:
    """Build a 3x3 'any material in cell' signature on the principal-axis
    aligned bbox. Returns (signature, fill_ratio)."""
    ang = _min_area_angle(outer)
    ro = _rotate(outer, ang)
    rh = [_rotate(h, ang) for h in holes]
    minx, miny, maxx, maxy = _bbox(ro)
    w = (maxx - minx) / 3.0
    h = (maxy - miny) / 3.0
    sub = 7  # sub-samples per cell per axis -> 49 probes; thin walls register
    rows = []
    filled_cells = 0
    total_probe = 0
    filled_probe = 0
    for r in range(3):           # top row first
        row = ""
        for c in range(3):
            cx0 = minx + c * w
            cy0 = miny + (2 - r) * h
            hit = False
            for i in range(sub):
                for jj in range(sub):
                    x = cx0 + (i + 0.5) / sub * w
                    y = cy0 + (jj + 0.5) / sub * h
                    total_probe += 1
                    if _point_in_region(x, y, ro, rh):
                        filled_probe += 1
                        hit = True
            row += "1" if hit else "0"
            if hit:
                filled_cells += 1
        rows.append(row)
    fill_ratio = filled_probe / total_probe if total_probe else 0.0
    return (rows[0], rows[1], rows[2]), fill_ratio


def _dihedral_orbit(sig: tuple[str, str, str]) -> set[tuple[str, str, str]]:
    """All 8 rotations/reflections of a 3x3 signature."""
    def as_grid(s):
        return [[int(s[r][c]) for c in range(3)] for r in range(3)]

    def to_sig(g):
        return tuple("".join(str(g[r][c]) for c in range(3)) for r in range(3))

    def rot90(g):
        return [[g[2 - c][r] for c in range(3)] for r in range(3)]

    def flip(g):
        return [[g[r][2 - c] for c in range(3)] for r in range(3)]

    g = as_grid(sig)
    out = set()
    for _ in range(4):
        out.add(to_sig(g))
        out.add(to_sig(flip(g)))
        g = rot90(g)
    return out


def _match_signature(sig: tuple[str, str, str]) -> tuple[SectionType, float]:
    """Match an observed signature against canonical templates over the
    dihedral group. Confidence reflects exact vs near match."""
    orbit = _dihedral_orbit(sig)
    for stype, canon in _CANONICAL.items():
        if canon in orbit:
            return stype, 0.9
    # no exact match: nearest by Hamming distance over the orbit
    best, best_d = SectionType.UNKNOWN, 99
    flat_obs = ["".join(s) for s in orbit]
    for stype, canon in _CANONICAL.items():
        canon_flat = "".join(canon)
        d = min(sum(a != b for a, b in zip(o, canon_flat)) for o in flat_obs)
        if d < best_d:
            best, best_d = stype, d
    conf = max(0.0, 1.0 - best_d / 9.0) * 0.6
    return best, conf


# ---------------------------------------------------------------------------
# lip detection for channels
# ---------------------------------------------------------------------------

def _detect_lips(outer: Loop, depth: float, width: float) -> bool:
    """A lipped channel has short return segments at the flange tips,
    giving extra vertices on the open side. Heuristic: a plain channel
    outline has ~8 vertices, a lipped one ~12. We also require the extra
    vertices to be short segments (the returns)."""
    # de-duplicate near-coincident points
    pts = list(outer)
    cleaned = [pts[0]]
    for p in pts[1:]:
        if math.hypot(p[0] - cleaned[-1][0], p[1] - cleaned[-1][1]) > 1e-6:
            cleaned.append(p)
    return len(cleaned) >= 11


# ---------------------------------------------------------------------------
# public entry point
# ---------------------------------------------------------------------------

def classify_section(
    loops: list[Loop],
    detection_method: str = "extrude_profile",
) -> CrossSection:
    """Classify a cross-section from its profile loops.

    loops[0] is the outer boundary; any further loops are holes (voids).
    Points are 2D (u, v) in the profile plane, arcs already tessellated.
    """
    if not loops or len(loops[0]) < 3:
        return CrossSection(detection_method=detection_method, confidence=0.0)

    outer = loops[0]
    holes = [lp for lp in loops[1:] if len(lp) >= 3]
    n_loops = 1 + len(holes)
    minx, miny, maxx, maxy = _bbox(outer)
    bb_w, bb_h = maxx - minx, maxy - miny
    depth, width = max(bb_w, bb_h), min(bb_w, bb_h)

    cs = CrossSection(detection_method=detection_method, n_loops=n_loops,
                      depth=depth, width=width)

    # 1) round family via circle fit on the outer loop.
    # Guard: a rectangle's four corners are concyclic and pass a pure
    # radius fit, so also require the enclosed area to match a disc.
    cx, cy, r_out, err_out = _fit_circle(outer)
    area_out = _polygon_area(outer)
    disc_ratio = area_out / (math.pi * r_out * r_out) if r_out > 1e-9 else 0.0
    if err_out < 0.04 and 0.85 <= disc_ratio <= 1.15:
        if holes:
            _, _, _, err_in = _fit_circle(holes[0])
            if err_in < 0.06:
                cs.section_type = SectionType.ROUND_HSS
                cs.family = SectionFamily.HOLLOW
                cs.confidence = 0.9
                return cs
        cs.section_type = SectionType.ROUND_BAR
        cs.family = SectionFamily.OPEN
        cs.confidence = 0.88
        return cs

    # 2) occupancy signature for prismatic / angular profiles
    sig, fill = _occupancy(outer, holes)
    cs.occupancy_signature = "/".join(sig)
    stype, conf = _match_signature(sig)

    # disambiguate hollow vs solid using loop count + fill
    if stype == SectionType.RECT_HSS and n_loops < 2:
        # 111/101/111 with no real void is unusual -> treat as plate-ish
        stype, conf = SectionType.PLATE, conf * 0.7
    if stype == SectionType.PLATE and fill < 0.9 and n_loops >= 2:
        stype = SectionType.RECT_HSS

    cs.section_type = stype
    cs.family = SectionFamily.HOLLOW if stype in HOLLOW_TYPES else SectionFamily.OPEN
    cs.confidence = round(conf, 2)

    # 3) channel lip refinement + thin-wall thickness estimate
    if stype == SectionType.CHANNEL:
        cs.is_lipped = _detect_lips(outer, depth, width)
    if cs.family == SectionFamily.OPEN and fill < 0.85 and depth > 0:
        # crude thin-wall thickness from area share of the bounding box
        area = _polygon_area(outer) - sum(_polygon_area(h) for h in holes)
        perim = _perimeter_estimate(stype, depth, width)
        if perim > 0:
            cs.wall_thickness = round(area / perim, 3)

    return cs


def _polygon_area(loop: Loop) -> float:
    a = 0.0
    n = len(loop)
    for i in range(n):
        x1, y1 = loop[i]
        x2, y2 = loop[(i + 1) % n]
        a += x1 * y2 - x2 * y1
    return abs(a) / 2.0


def _perimeter_estimate(stype: SectionType, depth: float, width: float) -> float:
    """Approximate centreline perimeter used to back out a wall thickness."""
    if stype == SectionType.CHANNEL:
        return depth + 2 * width
    if stype == SectionType.I_BEAM:
        return depth + 2 * width
    if stype == SectionType.ANGLE:
        return depth + width
    if stype == SectionType.TEE:
        return depth + width
    return 2 * (depth + width)


## 3. Joint detector

Finds where member centrelines come close enough to connect, records whether each member ends there or passes through, and clusters contacts into joint nodes. The looser `tol_infer` band is the "where it *should* go" case → flagged `is_inferred`.

In [13]:
"""Joint detector.

Operates on Member centrelines (start/end points in assembly space). It
finds where members come close enough to be connected, records for each
member whether the contact is at one of its ends or along its span, and
clusters nearby contacts into joint nodes.

Two tolerances:
  tol_touch  -- members this close are physically connected   ("is there")
  tol_infer  -- members within this looser band are treated as an intended
                connection with a small gap                   ("should go")

This module is pure geometry, so it is testable without Inventor by
handing it synthetic Member objects.
"""

from __future__ import annotations

import math
from dataclasses import dataclass, field



@dataclass
class _Contact:
    member_index: int
    at_end: bool          # True: contact at a member end; False: along the span
    direction: tuple[float, float, float]  # unit axis pointing AWAY from node


@dataclass
class JointCandidate:
    location: tuple[float, float, float]
    contacts: list[_Contact] = field(default_factory=list)
    is_inferred: bool = False


# --- small vector helpers ---------------------------------------------------

def _sub(a, b): return (a[0] - b[0], a[1] - b[1], a[2] - b[2])
def _add(a, b): return (a[0] + b[0], a[1] + b[1], a[2] + b[2])
def _scale(a, s): return (a[0] * s, a[1] * s, a[2] * s)
def _dot(a, b): return a[0] * b[0] + a[1] * b[1] + a[2] * b[2]
def _norm(a): return math.sqrt(_dot(a, a))


def _unit(a):
    n = _norm(a)
    return (a[0] / n, a[1] / n, a[2] / n) if n > 1e-12 else (0.0, 0.0, 0.0)


def _closest_params(p1, p2, q1, q2):
    """Closest points between segments p1p2 and q1q2.
    Returns (dist, s, t) with s, t in [0, 1]."""
    d1 = _sub(p2, p1)
    d2 = _sub(q2, q1)
    r = _sub(p1, q1)
    a = _dot(d1, d1)
    e = _dot(d2, d2)
    f = _dot(d2, r)
    if a < 1e-12 and e < 1e-12:
        return _norm(r), 0.0, 0.0
    if a < 1e-12:
        s = 0.0
        t = min(max(f / e, 0.0), 1.0)
    else:
        c = _dot(d1, r)
        if e < 1e-12:
            t = 0.0
            s = min(max(-c / a, 0.0), 1.0)
        else:
            b = _dot(d1, d2)
            denom = a * e - b * b
            s = min(max((b * f - c * e) / denom, 0.0), 1.0) if denom > 1e-12 else 0.0
            t = (b * s + f) / e
            if t < 0.0:
                t = 0.0
                s = min(max(-c / a, 0.0), 1.0)
            elif t > 1.0:
                t = 1.0
                s = min(max((b - c) / a, 0.0), 1.0)
    cp = _add(p1, _scale(d1, s))
    cq = _add(q1, _scale(d2, t))
    return _norm(_sub(cp, cq)), s, t


def _away_direction(member: Member, param: float):
    """Unit vector from the contact point toward the member's far end."""
    s = member.start_point
    e = member.end_point
    contact = _add(s, _scale(_sub(e, s), param))
    far = e if param < 0.5 else s
    return _unit(_sub(far, contact))


def detect_joints(
    members: list[Member],
    tol_touch: float = 0.2,
    tol_infer: float = 2.0,
    end_frac: float = 0.05,
    cluster_tol: float = 1.0,
) -> list[JointCandidate]:
    """Find joint candidates among member centrelines.

    end_frac: a contact within this fraction of a member's length from an
    end counts as an end contact (otherwise it is a span/through contact).
    """
    raw = []  # (location, contact_a, contact_b, inferred)
    for i in range(len(members)):
        for j in range(i + 1, len(members)):
            mi, mj = members[i], members[j]
            dist, s, t = _closest_params(
                mi.start_point, mi.end_point, mj.start_point, mj.end_point)
            if dist > tol_infer:
                continue
            inferred = dist > tol_touch
            pi = _add(mi.start_point, _scale(_sub(mi.end_point, mi.start_point), s))
            pj = _add(mj.start_point, _scale(_sub(mj.end_point, mj.start_point), t))
            loc = _scale(_add(pi, pj), 0.5)
            ci = _Contact(i, (s <= end_frac or s >= 1 - end_frac), _away_direction(mi, s))
            cj = _Contact(j, (t <= end_frac or t >= 1 - end_frac), _away_direction(mj, t))
            raw.append((loc, ci, cj, inferred))

    # cluster raw contacts into joint nodes by location proximity
    joints: list[JointCandidate] = []
    for loc, ci, cj, inferred in raw:
        host = None
        for jc in joints:
            if _norm(_sub(jc.location, loc)) <= cluster_tol:
                host = jc
                break
        if host is None:
            host = JointCandidate(location=loc)
            joints.append(host)
        host.is_inferred = host.is_inferred or inferred
        for c in (ci, cj):
            if not any(existing.member_index == c.member_index for existing in host.contacts):
                host.contacts.append(c)
    return joints


## 4. Joint classifier

Topology-first. The taxonomy is chosen by section family: all-hollow nodes get tubular names (`T/Y/X/K/N`), open-section nodes get connection-configuration names (`splice/corner/tee_connection/crossing/gusset`). Open sections are never forced into tubular nomenclature.

In [14]:
"""Joint classifier (topology-first).

Given a joint candidate (the members meeting at a node, each flagged as
ending there or passing through, plus their directions away from the
node), decide the joint type.

The taxonomy is chosen by the section family of the members, and the two
taxonomies are kept strictly separate so an open-section connection is
never mislabelled with tubular nomenclature:

  * All members hollow  -> tubular taxonomy (CIDECT / Eurocode 3):
        T, Y, X, K, N
  * Otherwise           -> open-section connection configuration:
        splice, corner, tee_connection, crossing, gusset

"Topology-first" means we first read the structure of the node -- how
many members, which pass through, the angles between them -- and only
then apply names. The same topology routine feeds both taxonomies.
"""

from __future__ import annotations

import math



def _angle_deg(a, b) -> float:
    d = max(-1.0, min(1.0, _dot(a, b)))
    return math.degrees(math.acos(d))


def _pairwise_angles(dirs) -> list[float]:
    out = []
    for i in range(len(dirs)):
        for j in range(i + 1, len(dirs)):
            out.append(_angle_deg(dirs[i], dirs[j]))
    return out


def classify_joint(members: list[Member], cand: JointCandidate, jid: str) -> Joint:
    contacts = cand.contacts
    names = [members[c.member_index].name if hasattr(members[c.member_index], "name")
             else members[c.member_index].occurrence_name for c in contacts]
    dirs = [c.direction for c in contacts]
    angles = [round(a, 1) for a in _pairwise_angles(dirs)]

    all_hollow = all(
        members[c.member_index].cross_section.family == SectionFamily.HOLLOW
        for c in contacts
    ) and len(contacts) > 0

    joint = Joint(
        joint_id=jid,
        location=tuple(round(x, 4) for x in cand.location),
        member_names=names,
        angles_deg=angles,
        is_inferred=cand.is_inferred,
        taxonomy="tubular" if all_hollow else "open",
    )

    through = [c for c in contacts if not c.at_end]
    ending = [c for c in contacts if c.at_end]
    n = len(contacts)

    if all_hollow:
        _classify_tubular(joint, members, through, ending, contacts)
    else:
        _classify_open(joint, members, through, ending, contacts, angles)

    return joint


def _classify_open(joint, members, through, ending, contacts, angles):
    n = len(contacts)
    if n >= 3:
        joint.joint_type = JointType.GUSSET
        joint.confidence = 0.6
        for c in contacts:
            joint.member_roles[_name(members, c)] = "leg"
        return
    if n == 2:
        if len(through) == 2:
            joint.joint_type = JointType.CROSSING
            joint.confidence = 0.7
        elif len(through) == 1:
            joint.joint_type = JointType.TEE_CONNECTION
            joint.confidence = 0.75
            joint.member_roles[_name(members, through[0])] = "continuous"
            joint.member_roles[_name(members, ending[0])] = "branch"
        else:  # both ending
            ang = angles[0] if angles else 0.0
            if ang >= 150:
                joint.joint_type = JointType.SPLICE
            else:
                joint.joint_type = JointType.CORNER
            joint.confidence = 0.75
        return
    joint.joint_type = JointType.UNKNOWN
    joint.confidence = 0.3


def _classify_tubular(joint, members, through, ending, contacts):
    # chord: a through member if present, else the longest member at the node
    if through:
        chord = through[0]
    else:
        chord = max(contacts, key=lambda c: members[c.member_index].length)
    braces = [c for c in contacts if c is not chord and c.at_end]
    joint.member_roles[_name(members, chord)] = "chord"
    for b in braces:
        joint.member_roles[_name(members, b)] = "brace"

    chord_dir = chord.direction
    nb = len(braces)
    if nb == 1:
        ang = _angle_deg(braces[0].direction, chord_dir)
        ang = min(ang, 180 - ang)            # angle to the chord axis
        joint.joint_type = JointType.T if abs(ang - 90) <= 15 else JointType.Y
        joint.confidence = 0.7
    elif nb == 2:
        a0 = _angle_deg(braces[0].direction, chord_dir)
        a1 = _angle_deg(braces[1].direction, chord_dir)
        between = _angle_deg(braces[0].direction, braces[1].direction)
        # opposite sides of the chord and roughly collinear -> X
        if between >= 150:
            joint.joint_type = JointType.X
            joint.confidence = 0.65
        else:
            perp0 = abs(min(a0, 180 - a0) - 90) <= 15
            perp1 = abs(min(a1, 180 - a1) - 90) <= 15
            joint.joint_type = JointType.N if (perp0 ^ perp1) else JointType.K
            joint.confidence = 0.6
    else:
        joint.joint_type = JointType.K   # multi-brace; refine later
        joint.confidence = 0.4


def _name(members, contact):
    m = members[contact.member_index]
    return m.occurrence_name


def classify_all(members: list[Member], candidates: list[JointCandidate]) -> list[Joint]:
    return [classify_joint(members, c, f"J{idx+1:03d}")
            for idx, c in enumerate(candidates)]


## 5. Inventor session (Windows only)

The only module that talks to Inventor's COM server. On non-Windows / without pywin32 it stays dormant — the cell still defines the class so the notebook runs everywhere; it only raises if you actually try to connect.

In [15]:
"""Inventor connection layer (COM via pywin32).

WINDOWS ONLY. Requires Autodesk Inventor installed and pywin32
(`pip install pywin32`). This is the only module that talks to Inventor's
COM server; everything downstream works on plain Python objects.

Usage:
    with InventorSession() as inv:
        asm = inv.active_assembly()
        ...
"""

from __future__ import annotations

try:
    import win32com.client as _w32          # type: ignore
    import pythoncom                         # type: ignore
    _HAVE_WIN32 = True
except Exception:                            # not on Windows / no pywin32
    _HAVE_WIN32 = False


class InventorError(RuntimeError):
    pass


class InventorSession:
    """Attaches to a running Inventor instance, or launches one."""

    def __init__(self, launch_if_absent: bool = False, visible: bool = True):
        if not _HAVE_WIN32:
            raise InventorError(
                "pywin32 not available. This module must run on Windows with "
                "Autodesk Inventor and `pip install pywin32`."
            )
        self.launch_if_absent = launch_if_absent
        self.visible = visible
        self.app = None

    def __enter__(self):
        self.connect()
        return self

    def __exit__(self, *exc):
        # We do not close Inventor we did not launch; just drop the handle.
        self.app = None

    def connect(self):
        try:
            self.app = _w32.GetActiveObject("Inventor.Application")
        except Exception:
            if not self.launch_if_absent:
                raise InventorError(
                    "No running Inventor instance found. Open Inventor with the "
                    "model loaded, or construct with launch_if_absent=True."
                )
            self.app = _w32.Dispatch("Inventor.Application")
            self.app.Visible = self.visible
        return self.app

    def active_document(self):
        if self.app is None:
            self.connect()
        doc = self.app.ActiveDocument
        if doc is None:
            raise InventorError("No active document is open in Inventor.")
        return doc

    def active_assembly(self):
        """Return the active document, asserting it is an assembly.

        DocumentType 12291 == kAssemblyDocumentObject.
        """
        doc = self.active_document()
        if int(doc.DocumentType) != 12291:
            raise InventorError(
                "Active document is not an assembly. Open the HVAC assembly "
                "(.iam) as the active document."
            )
        return doc


## 6. GLV BOM filter

Walks the assembly occurrences (recursively), reads each part's Description from the *Design Tracking Properties* set (what populates the BOM Description column), and keeps the ones containing `GLV`.

In [16]:
"""GLV member filter.

Walks the assembly's component occurrences (recursively through
sub-assemblies), reads each part's Description, and keeps only the ones
whose description contains the GLV tag -- the galvanised structural
members. Everything else (fasteners, sheet panels, purchased parts)
drops out here, which is the cheapest, highest-leverage filter in the
pipeline.

Description and other identifiers come from the "Design Tracking
Properties" iProperty set, which is exactly what populates the BOM
Description column, so this honours the "identify members via GLV in the
BOM description" requirement.

Inventor-dependent: must run inside an InventorSession.
"""

from __future__ import annotations


_DESIGN_TRACKING = "Design Tracking Properties"


def _prop(definition_doc, name: str, default: str = "") -> str:
    """Read a Design Tracking Property value, tolerating absence."""
    try:
        ps = definition_doc.PropertySets.Item(_DESIGN_TRACKING)
        return str(ps.Item(name).Value or default)
    except Exception:
        return default


def _iter_occurrences(occurrences):
    """Yield every leaf part occurrence, descending into sub-assemblies.

    IMPORTANT: Inventor COM collections are 1-INDEXED. Iterating one with a
    plain ``for occ in occurrences`` makes pywin32 fall back to 0-indexed
    item access (``Item(0)``), which Inventor rejects with a generic
    com_error (-2147352567, 'Exception occurred'). Always index explicitly
    from 1 with ``.Item(i)``. Each COM access is also guarded so a single
    unresolved/suppressed occurrence cannot abort the whole walk.
    """
    try:
        count = occurrences.Count
    except Exception:
        return
    for i in range(1, count + 1):
        try:
            occ = occurrences.Item(i)
        except Exception:
            continue
        try:
            subs = occ.SubOccurrences
            has_subs = subs is not None and subs.Count > 0
        except Exception:
            subs, has_subs = None, False
        if has_subs:
            yield from _iter_occurrences(subs)
        else:
            yield occ


def filter_glv_members(assembly_doc, tag: str = "GLV") -> list[Member]:
    """Return one Member per GLV-tagged occurrence (geometry filled later)."""
    tag = tag.upper()
    members: list[Member] = []
    asm_def = assembly_doc.ComponentDefinition
    for occ in _iter_occurrences(asm_def.Occurrences):
        try:
            def_doc = occ.Definition.Document
            desc = _prop(def_doc, "Description")
        except Exception:
            continue                      # suppressed / unresolved / virtual
        if tag not in desc.upper():
            continue
        try:
            m = Member(
                occurrence_name=str(occ.Name),
                bom_description=desc,
                part_number=_prop(def_doc, "Part Number"),
                material=_prop(def_doc, "Material"),
                is_glv=True,
                cross_section=CrossSection(),
            )
            m._occ = occ          # live COM handle, consumed by member_geometry
            members.append(m)
        except Exception:
            continue
    return members


## 7. Member geometry

Centreline from the body range box's longest axis; cross-section loops read straight from the dominant extrude's profile sketch and handed to the classifier from section 2.

In [17]:
"""Member geometry extraction.

For each GLV member this fills in:
  * the centreline (start/end points) in assembly coordinates, and
  * the cross-section profile loops, read directly from the member's
    extrude feature -- because the model uses plain extrudes, the
    extrude's profile sketch IS the cross-section, so we never have to
    slice a solid.

The profile loops are handed to section_classifier.classify_section,
keeping geometry acquisition and shape recognition decoupled (the
classifier is tested independently against synthetic loops).

Inventor-dependent. Units are Inventor internal (centimetres); the
classifier is unit-agnostic so that is harmless.

ASSUMPTIONS TO VERIFY against the real model:
  * each structural member is a single dominant extrude;
  * the extrude profile's largest path is the outer boundary;
  * end copes/mitres are separate cut features and do not alter the
    base extrude profile (true for typical "extrude then cut" modelling).
If a member is built differently, classify_section still runs but its
confidence will drop -- inspect occupancy_signature to see what it saw.
"""

from __future__ import annotations

import math



# --- centreline ------------------------------------------------------------

def _transform_point(matrix, p):
    """Apply an Inventor Matrix to a (x,y,z) tuple. Inventor matrices are
    4x4 with .Cell(row, col), 1-indexed."""
    x, y, z = p
    out = []
    for r in range(1, 4):
        out.append(
            matrix.Cell(r, 1) * x + matrix.Cell(r, 2) * y
            + matrix.Cell(r, 3) * z + matrix.Cell(r, 4)
        )
    return tuple(out)


def _centerline(occ, part_def):
    """Centreline endpoints in assembly space from the body range box and
    the dominant extrude axis. The axis is the longest dimension of the
    box; endpoints are the box centre offset by +/- half that length."""
    body = part_def.SurfaceBodies.Item(1)
    rb = body.RangeBox
    mn = (rb.MinPoint.X, rb.MinPoint.Y, rb.MinPoint.Z)
    mx = (rb.MaxPoint.X, rb.MaxPoint.Y, rb.MaxPoint.Z)
    dims = [mx[i] - mn[i] for i in range(3)]
    axis = max(range(3), key=lambda i: dims[i])      # longest dimension
    center = [(mn[i] + mx[i]) / 2.0 for i in range(3)]
    half = dims[axis] / 2.0
    s = list(center); s[axis] -= half
    e = list(center); e[axis] += half
    m = occ.Transformation
    s = _transform_point(m, tuple(s))
    e = _transform_point(m, tuple(e))
    length = math.dist(s, e)
    return s, e, length


# --- cross-section loops from the extrude profile --------------------------

def _tessellate_arc(arc, n: int = 10):
    """Sample an Inventor SketchArc into points in sketch (u,v) space."""
    c = arc.CenterSketchPoint.Geometry
    r = arc.Radius
    a0 = arc.StartAngle
    sweep = arc.SweepAngle
    pts = []
    for i in range(n + 1):
        a = a0 + sweep * i / n
        pts.append((c.X + r * math.cos(a), c.Y + r * math.sin(a)))
    return pts


def _path_points(path):
    """Ordered (u,v) points for one ProfilePath (a closed loop)."""
    pts = []
    for i in range(1, path.Count + 1):
        entity = path.Item(i)
        se = entity.SketchEntity
        kind = se.Type
        # SketchLine
        if "Line" in str(kind) or hasattr(se, "StartSketchPoint") and not hasattr(se, "Radius"):
            g0 = se.StartSketchPoint.Geometry
            pts.append((g0.X, g0.Y))
        # SketchArc
        elif hasattr(se, "Radius") and hasattr(se, "SweepAngle"):
            pts.extend(_tessellate_arc(se))
        # SketchCircle (full circle) -> sample
        elif hasattr(se, "Radius"):
            c = se.CenterSketchPoint.Geometry
            r = se.Radius
            pts.extend((c.X + r * math.cos(2 * math.pi * k / 32),
                        c.Y + r * math.sin(2 * math.pi * k / 32)) for k in range(32))
    return pts


def _loop_area(pts):
    a = 0.0
    n = len(pts)
    for i in range(n):
        x1, y1 = pts[i]
        x2, y2 = pts[(i + 1) % n]
        a += x1 * y2 - x2 * y1
    return abs(a) / 2.0


def _section_loops(part_def):
    """Return profile loops [outer, *holes] from the dominant extrude."""
    extrudes = part_def.Features.ExtrudeFeatures
    if extrudes.Count == 0:
        return []
    # dominant extrude = the one whose profile encloses the most area
    best, best_area = None, -1.0
    for i in range(1, extrudes.Count + 1):
        ext = extrudes.Item(i)
        try:
            prof = ext.Profile
            total = sum(_loop_area(_path_points(prof.Item(k)))
                        for k in range(1, prof.Count + 1))
        except Exception:
            total = -1.0
        if total > best_area:
            best, best_area = ext, total
    if best is None:
        return []
    prof = best.Profile
    loops = [_path_points(prof.Item(k)) for k in range(1, prof.Count + 1)]
    loops = [lp for lp in loops if len(lp) >= 3]
    loops.sort(key=_loop_area, reverse=True)     # largest first = outer
    return loops


# --- public ----------------------------------------------------------------

def fill_geometry(members: list[Member]) -> list[Member]:
    """Populate centreline and cross_section for each member, in place.
    Each Member must carry a live `._occ` handle set by the caller (see
    main.py) -- we keep COM handles out of the dataclass fields so the
    result stays JSON-serialisable."""
    for m in members:
        occ = getattr(m, "_occ", None)
        if occ is None:
            continue
        part_def = occ.Definition
        try:
            s, e, length = _centerline(occ, part_def)
            m.start_point, m.end_point, m.length = s, e, length
        except Exception:
            pass
        try:
            loops = _section_loops(part_def)
            if loops:
                m.cross_section = classify_section(loops, detection_method="extrude_profile")
        except Exception:
            pass
    return members


## 8. Orchestrator

In [18]:
"""8. Orchestrator — run the full Block 1 pipeline against the active assembly."""
import sys


def run(tag: str = "GLV", tol_touch: float = 0.2, tol_infer: float = 2.0) -> AnalysisResult:
    with InventorSession() as inv:
        asm = inv.active_assembly()
        result = AnalysisResult(source_document=str(asm.FullFileName))

        members = filter_glv_members(asm, tag=tag)
        members = fill_geometry(members)
        result.members = members

        candidates = detect_joints(members, tol_touch=tol_touch, tol_infer=tol_infer)
        result.joints = classify_all(members, candidates)

    for m in result.members:          # drop live COM handles before serialising
        if hasattr(m, "_occ"):
            del m._occ
    return result


def summarise(result: AnalysisResult) -> None:
    n_inf = sum(1 for j in result.joints if j.is_inferred)
    print(f"Members (GLV): {len(result.members)}")
    print(f"Joints:        {len(result.joints)}  ({n_inf} inferred / 'should go')")
    by_type = {}
    for j in result.joints:
        by_type[j.joint_type.value] = by_type.get(j.joint_type.value, 0) + 1
    for k, v in sorted(by_type.items()):
        print(f"  {k:16} {v}")


## 9. Tests & demo 

This builds synthetic profiles for every section type and synthetic members for every joint configuration, runs them through the same code the live pipeline uses, prints a results table, and asserts correctness. Run this cell to confirm the analytical core works on your machine.

In [19]:
import math

# --- synthetic 2D section profiles -----------------------------------------
def _plate(w=80.0, h=20.0):
    return [[(0, 0), (w, 0), (w, h), (0, h)]]
def _round_bar(r=25.0, n=48):
    return [[(r*math.cos(2*math.pi*i/n), r*math.sin(2*math.pi*i/n)) for i in range(n)]]
def _round_hss(ro=40.0, ri=33.0, n=48):
    return [[(ro*math.cos(2*math.pi*i/n), ro*math.sin(2*math.pi*i/n)) for i in range(n)],
            [(ri*math.cos(2*math.pi*i/n), ri*math.sin(2*math.pi*i/n)) for i in range(n)]]
def _rect_hss(w=80.0, h=120.0, t=6.0):
    return [[(0,0),(w,0),(w,h),(0,h)], [(t,t),(w-t,t),(w-t,h-t),(t,h-t)]]
def _plain_channel(d=120.0, w=50.0, t=6.0):
    return [[(0,0),(w,0),(w,t),(t,t),(t,d-t),(w,d-t),(w,d),(0,d)]]
def _lipped_channel(d=120.0, w=50.0, t=4.0, lip=12.0):
    return [[(0,0),(w,0),(w,lip),(w-t,lip),(w-t,t),(t,t),
             (t,d-t),(w-t,d-t),(w-t,d-lip),(w,d-lip),(w,d),(0,d)]]
def _angle(L=100.0, t=8.0):
    return [[(0,0),(L,0),(L,t),(t,t),(t,L),(0,L)]]
def _i_beam(d=120.0, w=70.0, tf=8.0, tw=6.0):
    x0,x1=(w-tw)/2,(w+tw)/2
    return [[(0,0),(w,0),(w,tf),(x1,tf),(x1,d-tf),(w,d-tf),(w,d),(0,d),
             (0,d-tf),(x0,d-tf),(x0,tf),(0,tf)]]
def _tee(d=100.0, w=100.0, tf=10.0, tw=10.0):
    x0,x1=(w-tw)/2,(w+tw)/2
    return [[(0,d-tf),(w,d-tf),(w,d),(0,d),(0,d-tf),(x0,d-tf),(x0,0),(x1,0),(x1,d-tf)]]

_SECTIONS = {
    "plate": (_plate, "plate"), "round_bar": (_round_bar, "round_bar"),
    "round_hss": (_round_hss, "round_hss"), "rect_hss": (_rect_hss, "rect_hss"),
    "plain_channel": (_plain_channel, "channel"), "lipped_channel": (_lipped_channel, "channel"),
    "angle": (_angle, "angle"), "i_beam": (_i_beam, "i_beam"), "tee": (_tee, "tee"),
}

def _rot(loops, deg):
    a = math.radians(deg); c, s = math.cos(a), math.sin(a)
    return [[(x*c - y*s, x*s + y*c) for (x, y) in lp] for lp in loops]

print("SECTION CLASSIFIER  (profiles rotated 37 deg to prove orientation-independence)")
print(f"{'profile':16}{'expected':11}{'got':11}{'conf':6}{'signature':13}{'lip'}")
print("-"*64)
sec_ok = 0
for name, (fn, expected) in _SECTIONS.items():
    cs = classify_section(_rot(fn(), 37))
    lip = "" if cs.is_lipped is None else ("yes" if cs.is_lipped else "no")
    mark = "OK" if cs.section_type.value == expected else "XX"
    sec_ok += cs.section_type.value == expected
    print(f"{name:16}{expected:11}{cs.section_type.value:11}{cs.confidence:<6}{str(cs.occupancy_signature or ''):13}{lip:4}{mark}")
print(f"-> {sec_ok}/{len(_SECTIONS)} correct\n")

# --- synthetic joints ------------------------------------------------------
def _mk(name, s, e, hollow=False):
    fam = SectionFamily.HOLLOW if hollow else SectionFamily.OPEN
    st  = SectionType.RECT_HSS if hollow else SectionType.CHANNEL
    return Member(occurrence_name=name, is_glv=True, start_point=s, end_point=e,
                  cross_section=CrossSection(section_type=st, family=fam))

_SCENARIOS = {
    "open tee":     ([_mk("A",(0,0,0),(100,0,0)), _mk("B",(50,0,0),(50,50,0))], "tee_connection"),
    "open corner":  ([_mk("A",(0,0,0),(100,0,0)), _mk("B",(100,0,0),(100,80,0))], "corner"),
    "open splice":  ([_mk("A",(0,0,0),(50,0,0)),  _mk("B",(50,0,0),(100,0,0))], "splice"),
    "hollow T":     ([_mk("A",(0,0,0),(100,0,0),1), _mk("B",(50,0,0),(50,50,0),1)], "T"),
    "hollow Y":     ([_mk("A",(0,0,0),(100,0,0),1), _mk("B",(50,0,0),(80,40,0),1)], "Y"),
    "gusset (3)":   ([_mk("A",(0,0,0),(50,0,0)), _mk("B",(50,0,0),(80,40,0)), _mk("C",(50,0,0),(80,-40,0))], "gusset"),
    "inferred gap": ([_mk("A",(0,0,0),(100,0,0)), _mk("B",(50,1.0,0),(50,50,0))], "tee_connection"),
}

print("JOINT DETECTOR + CLASSIFIER")
print(f"{'scenario':14}{'type':16}{'taxonomy':10}{'inferred':10}{'expected'}")
print("-"*62)
jt_ok = 0
for name, (members, expected) in _SCENARIOS.items():
    joints = classify_all(members, detect_joints(members))
    j = joints[0]
    mark = "OK" if j.joint_type.value == expected else "XX"
    jt_ok += j.joint_type.value == expected
    print(f"{name:14}{j.joint_type.value:16}{j.taxonomy:10}{str(j.is_inferred):10}{expected:10}{mark}")
print(f"-> {jt_ok}/{len(_SCENARIOS)} correct")

assert sec_ok == len(_SECTIONS), "section classifier regression"
assert jt_ok == len(_SCENARIOS), "joint classifier regression"
print()
print("All core tests passed.")


SECTION CLASSIFIER  (profiles rotated 37 deg to prove orientation-independence)
profile         expected   got        conf  signature    lip
----------------------------------------------------------------
plate           plate      plate      0.9   111/111/111      OK
round_bar       round_bar  round_bar  0.88                   OK
round_hss       round_hss  round_hss  0.9                    OK
rect_hss        rect_hss   rect_hss   0.9   111/101/111      OK
plain_channel   channel    channel    0.9   111/100/111  no  OK
lipped_channel  channel    channel    0.9   111/100/111  yes OK
angle           angle      angle      0.9   100/100/111      OK
i_beam          i_beam     i_beam     0.9   111/010/111      OK
tee             tee        tee        0.9   111/010/010      OK
-> 9/9 correct

JOINT DETECTOR + CLASSIFIER
scenario      type            taxonomy  inferred  expected
--------------------------------------------------------------
open tee      tee_connection  open      False     te

## 10. Run on the live model (Windows + Inventor)

Open the HVAC assembly as the **active document** in Inventor, `pip install pywin32`, then run the cell below. It filters GLV members, extracts geometry, detects and classifies joints, prints a summary, and writes `block1_result.json` — the contract for Blocks 2–4.

Tune `tol_touch` / `tol_infer` (centimetres) to your model. On non-Windows the cell prints a friendly message instead of erroring.

In [20]:
try:
    result = run(tag="GLV", tol_touch=0.2, tol_infer=2.0)
    summarise(result)
    with open("block1_result.json", "w", encoding="utf-8") as f:
        f.write(result.to_json())
    print("Wrote block1_result.json")
except InventorError as e:
    print(f"[Inventor not available] {e}")
except Exception as e:
    print(f"[Error during analysis] {type(e).__name__}: {e}")
    print("Re-run with the assembly active; if it persists, note which step failed.")


Members (GLV): 860
Joints:        540  (359 inferred / 'should go')
  T                3
  Y                22
  corner           105
  crossing         61
  gusset           53
  splice           52
  tee_connection   244
Wrote block1_result.json
